# Earnings NLP — FinBERT Signal Scoring
**Runtime → Change runtime type → T4 GPU** before running

In [ ]:
# Cell 1: Install
!pip install transformers torch sentence-transformers -q
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

GPU: Tesla T4


In [ ]:
# Cell 2: Upload your transcripts.db
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

Saving results.db to results.db
Saving transcripts.db to transcripts.db
Uploaded: ['results.db', 'transcripts.db']


In [ ]:
# Cell 3: Load FinBERT
import sqlite3, json, re
import numpy as np
from transformers import pipeline
from tqdm import tqdm

print('Loading ProsusAI/finbert...')
finbert_pipe = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    device=0 if torch.cuda.is_available() else -1,
    top_k=None,
)
print('FinBERT loaded')

def finbert_score(text):
    if not text or len(text.strip()) < 10:
        return {'positive': 0.33, 'negative': 0.33, 'neutral': 0.34, 'net': 0.0}
    words = text.split()
    if len(words) > 300:
        text = ' '.join(words[:300])
    try:
        result = finbert_pipe(text)[0]
        scores = {r['label']: r['score'] for r in result}
        net = scores.get('positive', 0) - scores.get('negative', 0)
        return {k: round(v, 4) for k, v in {**scores, 'net': net}.items()}
    except:
        return {'positive': 0.33, 'negative': 0.33, 'neutral': 0.34, 'net': 0.0}

# Quick test
test = finbert_score('Net interest margin compression is expected to persist through the year.')
print(f'Test: {test}')

Loading ProsusAI/finbert...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT loaded
Test: {'negative': 0.9544, 'positive': 0.0328, 'neutral': 0.0128, 'net': -0.9216}


In [ ]:
# Cell 4: Define all scoring functions
GUIDANCE_RE = re.compile(
    r'we expect|we anticipate|we project|we are targeting|we are guiding|'
    r'full.year|guidance|outlook|fiscal year|we forecast',
    re.IGNORECASE
)
NUMERIC_RE = re.compile(
    r'\b\d+(?:\.\d+)?(?:\s*(?:billion|million|%|percent|bps|basis points?))?\b'
    r'|\bbetween\s+\$?\d|\brange of\s+\$?\d',
    re.IGNORECASE
)
OPERATOR_NAMES = {'operator', 'moderator', 'coordinator'}

def extract_guidance_sentences(sc_json, raw_content=''):
    text = ''
    if sc_json:
        try:
            turns = json.loads(sc_json)
            if isinstance(turns, list):
                text = ' '.join(str(t.get('text','')) for t in turns)
        except: pass
    if not text and raw_content:
        text = raw_content
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences
            if len(s.strip()) > 30 and GUIDANCE_RE.search(s)][:25]

def score_guidance_finbert(sentences):
    if not sentences:
        return {'guidance_score': None, 'guidance_finbert_neg': None,
                'guidance_finbert_net': None, 'guidance_sentence_count': 0,
                'numeric_ratio': None}
    scores = [finbert_score(s) for s in sentences]
    avg_neg = np.mean([s['negative'] for s in scores])
    avg_net = np.mean([s['net']      for s in scores])
    num_ratio = sum(1 for s in sentences if NUMERIC_RE.search(s)) / len(sentences)
    evasiveness = (float(avg_neg) * 0.6) + ((1 - num_ratio) * 0.4)
    return {
        'guidance_score':         round(evasiveness, 4),
        'guidance_finbert_neg':   round(float(avg_neg), 4),
        'guidance_finbert_net':   round(float(avg_net), 4),
        'guidance_sentence_count': len(sentences),
        'numeric_ratio':           round(num_ratio, 4),
    }

def extract_sections(sc_json):
    if not sc_json: return {'prepared': [], 'qa_exec': []}
    try: turns = json.loads(sc_json)
    except: return {'prepared': [], 'qa_exec': []}
    prepared, qa_exec, prepared_speakers = [], [], set()
    in_qa = False
    for turn in turns:
        spk = str(turn.get('speaker','')).lower()
        txt = str(turn.get('text','')).strip()
        if any(op in spk for op in OPERATOR_NAMES): in_qa = True; continue
        if not in_qa and txt and len(txt.split()) > 20:
            prepared_speakers.add(spk); prepared.append(txt)
    in_qa = False; prev_analyst = False
    for turn in turns:
        spk = str(turn.get('speaker','')).lower()
        txt = str(turn.get('text','')).strip()
        if any(op in spk for op in OPERATOR_NAMES): in_qa=True; prev_analyst=False; continue
        if not in_qa: continue
        is_exec = spk in prepared_speakers
        if not is_exec: prev_analyst = True
        elif is_exec and prev_analyst and len(txt.split()) >= 30:
            qa_exec.append(txt); prev_analyst = False
    return {'prepared': prepared, 'qa_exec': qa_exec}

def score_sentiment_trajectory(sections):
    prepared = sections['prepared']
    qa_exec  = sections['qa_exec']
    if not prepared:
        return {'prepared_sentiment': None, 'qa_sentiment': None,
                'sentiment_drop': None, 'sentiment_trajectory': None}
    prep_scores = [finbert_score(c) for c in prepared[:5]]
    prep_net    = float(np.mean([s['net'] for s in prep_scores]))
    if not qa_exec:
        return {'prepared_sentiment': round(prep_net,4), 'qa_sentiment': None,
                'sentiment_drop': None, 'sentiment_trajectory': None}
    qa_scores = [finbert_score(c) for c in qa_exec[:8]]
    qa_net    = float(np.mean([s['net'] for s in qa_scores]))
    drop      = prep_net - qa_net
    if len(qa_scores) >= 3:
        h = len(qa_scores)//2
        traj = float(np.mean([s['net'] for s in qa_scores[:h]])) - \
               float(np.mean([s['net'] for s in qa_scores[h:]]))
    else: traj = 0.0
    return {'prepared_sentiment': round(prep_net,4), 'qa_sentiment': round(qa_net,4),
            'sentiment_drop': round(drop,4), 'sentiment_trajectory': round(traj,4)}

print('All scorers ready')

All scorers ready


In [ ]:
# Cell 5: Run on all transcripts
DB_PATH = 'transcripts.db'
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

existing = {row[1] for row in conn.execute('PRAGMA table_info(transcripts)')}
for col, dtype in [
    ('guidance_finbert_neg','REAL'), ('guidance_finbert_net','REAL'),
    ('prepared_sentiment','REAL'),   ('qa_sentiment','REAL'),
    ('sentiment_drop','REAL'),       ('sentiment_trajectory','REAL'),
]:
    if col not in existing:
        conn.execute(f'ALTER TABLE transcripts ADD COLUMN {col} {dtype}')
conn.commit()

rows = conn.execute(
    'SELECT id, symbol, year, quarter, structured_content, content FROM transcripts ORDER BY symbol, year, quarter'
).fetchall()
print(f'Processing {len(rows)} transcripts...')

for row in tqdm(rows):
    sentences = extract_guidance_sentences(row['structured_content'], row['content'] or '')
    g = score_guidance_finbert(sentences)
    sections = extract_sections(row['structured_content'])
    t = score_sentiment_trajectory(sections)
    conn.execute('''
        UPDATE transcripts SET
            guidance_score=?, guidance_finbert_neg=?, guidance_finbert_net=?,
            guidance_sentence_count=?, numeric_ratio=?,
            prepared_sentiment=?, qa_sentiment=?, sentiment_drop=?, sentiment_trajectory=?
        WHERE id=?
    ''', (g.get('guidance_score'), g.get('guidance_finbert_neg'), g.get('guidance_finbert_net'),
          g.get('guidance_sentence_count'), g.get('numeric_ratio'),
          t.get('prepared_sentiment'), t.get('qa_sentiment'),
          t.get('sentiment_drop'), t.get('sentiment_trajectory'), row['id']))

conn.commit()
conn.close()
print('Done!')

Processing 344 transcripts...


100%|██████████| 344/344 [01:14<00:00,  4.62it/s]

Done!


In [ ]:
# Cell 6: Validate results
conn = sqlite3.connect('transcripts.db')
rows = conn.execute('''
    SELECT symbol, year, quarter, guidance_score,
           prepared_sentiment, qa_sentiment, sentiment_drop
    FROM transcripts
    WHERE sentiment_drop IS NOT NULL
    ORDER BY sentiment_drop DESC LIMIT 10
''').fetchall()
print('Top 10 by sentiment drop:')
print(f'{"Ticker":<6} {"Year":<5} {"Q":<3} {"guidance":<10} {"prep_sent":<11} {"qa_sent":<9} drop')
for r in rows:
    print(f'{r[0]:<6} {r[1]:<5} {r[2]:<3} {r[3] or 0:<10.3f} {r[4] or 0:<11.3f} {r[5] or 0:<9.3f} {r[6] or 0:.3f}')
conn.close()

Top 10 by sentiment drop:
Ticker Year  Q   guidance   prep_sent   qa_sent   drop
PNC    2023  2   0.369      0.600       -0.110    0.710
PNC    2022  2   0.332      0.604       0.077     0.526
PNC    2022  4   0.234      0.524       0.052     0.472
PNC    2023  1   0.305      0.418       -0.044    0.463
PNC    2023  4   0.295      0.582       0.167     0.415
PNC    2022  3   0.297      0.180       -0.142    0.322
PNC    2023  3   0.348      -0.005      -0.163    0.158
PNC    2022  1   0.327      -0.057      -0.158    0.102
ZION   2024  4   0.262      0.404       0.357     0.047
KEY    2021  3   0.321      0.173       0.178     -0.005


In [ ]:
# Cell 7: Download updated DB
from google.colab import files
files.download('transcripts.db')
print('Replace your local data/transcripts.db with this file')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Replace your local data/transcripts.db with this file
